# Modelo de temperatura — Ley de enfriamiento de Newton

Elaborado por: *(nombre del estudiante)*

**Materia:** Ecuaciones Diferenciales

Este cuaderno modela cómo cambia la temperatura de un objeto que está en contacto con un ambiente a temperatura $T_a$.

$$\frac{dT}{dt} = -k\,(T - T_a), \qquad T(0) = T_0$$

- $T(t)$: temperatura del objeto en el tiempo $t$ (°C)
- $T_a$: temperatura del ambiente (°C)
- $k > 0$: constante de transferencia de calor (1/min)
- $T_0$: temperatura inicial del objeto (°C)

**Contenido**
1. Imports
2. Clase `Escenario` (EDO, solución exacta, Euler, equilibrio)
3. Punto de equilibrio y estabilidad
4. Casos de estudio y gráficas de series de tiempo
5. Campo de direcciones y línea de fase
6. Método de Euler vs. solución exacta (y estabilidad numérica)
7. Modelo modificado 1: temperatura ambiente variable
8. Modelo modificado 2: sistema acoplado objeto–ambiente
9. Variaciones de los parámetros ($k$, $T_a$, $T_0$) y mapas de calor
10. Modelo modificado 3: fuente de calor constante
11. Gráficas adicionales: escala semilog, constante de tiempo, RK4 y panel completo
12. Panel interactivo con deslizadores

![Modelo de temperatura](resources/Modelo-temperatura.png)

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp


plt.rcParams["figure.figsize"] = (10, 4)

## Parámetros globales

**Todo el cuaderno lee los valores de esta celda.** Para probar otro escenario cambia los números aquí y ejecuta *Run All*.
Las listas (`k_lista`, `Ta_lista`, ...) controlan las curvas de las gráficas de variaciones.

In [ ]:
# ============================ PARÁMETROS GLOBALES ============================

# --- Modelo base: dT/dt = -k (T - Ta) ---
k_base     = 0.15    # constante de transferencia de calor (1/min)
Ta_base    = 20.0    # temperatura del ambiente (°C)
T0_base    = 90.0    # objeto caliente (se enfría)
T0_frio    = 5.0     # objeto frío (se calienta)
k_alto     = 0.6     # k del "caso 4" y de la prueba de estabilidad de Euler
T_objetivo = 40.0    # temperatura a la que se quiere llegar (°C)

# --- Simulación ---
tiempo_final = 30    # min
paso         = 0.01  # paso de la simulación numérica (RK45)
h_euler      = 0.5   # paso para Euler y RK4 en los paneles

# --- Listas para las gráficas de variaciones ---
k_lista     = [0.05, 0.1, 0.2, 0.4, 0.8]
Ta_lista    = [0, 10, 20, 30, 50]
T0_lista    = [-10, 5, 20, 50, 90]
k_campos    = [0.05, 0.15, 0.4, 1.0]      # 4 valores para los campos de direcciones
h_lista     = [4.0, 2.0, 1.0, 0.25]       # pasos para comparar Euler
T_obj_lista = [70, 50, 40, 30]            # temperaturas objetivo (tiempo vs k)
t_mapa      = 10                          # instante del mapa de calor (min)

# --- Modelo modificado 1: ambiente variable ---
Tm_var, A_var, P_var = 20.0, 8.0, 24.0    # media, amplitud y periodo (h)
k_var, T0_var, tiempo_var = 0.5, 35.0, 72

# --- Modelo modificado 2: sistema acoplado objeto-aire-exterior ---
k1_ac, k2_ac, k3_ac, T_ext_ac = 0.3, 0.1, 0.05, 15.0
T0_ac, Ta0_ac, tiempo_ac = 90.0, 20.0, 80
cond_iniciales_ac = [(90, 20), (5, 20), (60, 5), (10, 40)]   # (T0, Ta0) del retrato de fase

# --- Modelo modificado 3: fuente de calor ---
q_base  = 3.0                              # °C/min
q_lista = [0, 1.5, 3.0, 4.5, 6.0]

# --- Límites automáticos de los ejes de temperatura ---
T_min_graf = min(T0_base, T0_frio, Ta_base, *T0_lista) - 10
T_max_graf = max(T0_base, T0_frio, Ta_base, *T0_lista) + 10

## 2. Clase `Escenario`

Agrupa los parámetros del modelo y los métodos para simular, calcular el equilibrio y graficar.

In [ ]:
class Escenario:
    """Ley de enfriamiento de Newton: dT/dt = -k (T - T_amb)."""

    def __init__(self, k=0.15, T_amb=20.0):
        # Parámetros del sistema
        self.k = k
        self.T_amb = T_amb

        # Símbolos algebraicos
        self.t_symb, self.T_symb = sp.symbols("t T", real=True)

        # Ecuación del sistema
        self.ecuacion = -self.k * (self.T_symb - self.T_amb)

    # ---------- EDO y soluciones ----------
    def edo(self, tiempo, T):
        return -self.k * (T - self.T_amb)

    def solucion_exacta(self, T0, t):
        """T(t) = T_amb + (T0 - T_amb) e^(-k t)"""
        return self.T_amb + (T0 - self.T_amb) * np.exp(-self.k * t)

    def solucion_simbolica(self, T0):
        """Resuelve la EDO con sympy (dsolve) para mostrar el procedimiento analítico."""
        T = sp.Function("T")
        eq = sp.Eq(T(self.t_symb).diff(self.t_symb),
                   -sp.nsimplify(self.k) * (T(self.t_symb) - sp.nsimplify(self.T_amb)))
        return sp.dsolve(eq, T(self.t_symb), ics={T(0): sp.nsimplify(T0)})

    def simular(self, T0, tiempo_final=30, paso=0.01):
        """Integra numéricamente con RK45 (scipy)."""
        self.T0 = T0
        self.tiempo_final = tiempo_final
        self.paso = paso

        num_puntos = int(tiempo_final / paso) + 1
        self.t_eval = np.linspace(0, tiempo_final, num_puntos)

        solucion = solve_ivp(self.edo, [0, tiempo_final], [T0],
                             t_eval=self.t_eval, method="RK45")
        self.tiempo, self.temperatura = solucion.t, solucion.y[0]
        return self.tiempo, self.temperatura

    def euler(self, T0, tiempo_final=30, h=0.5):
        """Método de Euler explícito: T_{n+1} = T_n + h f(t_n, T_n)."""
        n = int(round(tiempo_final / h))
        t = np.linspace(0, n * h, n + 1)
        T = np.zeros(n + 1)
        T[0] = T0
        for i in range(n):
            T[i + 1] = T[i] + h * self.edo(t[i], T[i])
        return t, T

    def T_equilibrio(self):
        """Temperatura de equilibrio del modelo (en el modelo base es T_amb)."""
        return self.T_amb

    def constante_de_tiempo(self):
        """tau = 1/k: tiempo en que se recorre el 63.2 % de la distancia al equilibrio."""
        return 1 / self.k

    def rk4(self, T0, tiempo_final=30, h=0.5):
        """Método de Runge-Kutta de orden 4 con paso fijo h."""
        n = int(round(tiempo_final / h))
        t = np.linspace(0, n * h, n + 1)
        T = np.zeros(n + 1)
        T[0] = T0
        for i in range(n):
            a1 = self.edo(t[i], T[i])
            a2 = self.edo(t[i] + h / 2, T[i] + h / 2 * a1)
            a3 = self.edo(t[i] + h / 2, T[i] + h / 2 * a2)
            a4 = self.edo(t[i] + h, T[i] + h * a3)
            T[i + 1] = T[i] + h / 6 * (a1 + 2 * a2 + 2 * a3 + a4)
        return t, T

    # ---------- Equilibrio y estabilidad ----------
    def calcular_puntos_equilibrio(self):
        puntos = sp.solve(self.ecuacion, self.T_symb, dict=True)
        jacobiano = sp.Matrix([self.ecuacion]).jacobian([self.T_symb])

        resultados = []
        for p in puntos:
            T_eq = float(p[self.T_symb])
            J = jacobiano.subs({self.T_symb: T_eq})
            autovalores = {round(float(ev), 2): mult for ev, mult in J.eigenvals().items()}
            lam = list(autovalores.keys())[0]
            if lam < 0:
                estabilidad = "asintóticamente estable (atractor)"
            elif lam > 0:
                estabilidad = "inestable (repulsor)"
            else:
                estabilidad = "no concluyente"
            resultados.append({
                "punto_equilibrio": round(T_eq, 2),
                "jacobiano": [[round(float(J[0, 0]), 2)]],
                "autovalores": autovalores,
                "estabilidad": estabilidad,
            })
        return resultados

    def tiempo_para_alcanzar(self, T0, T_objetivo):
        """Tiempo en que el objeto llega a T_objetivo (despeje de la solución exacta)."""
        razon = (T_objetivo - self.T_amb) / (T0 - self.T_amb)
        if razon <= 0 or razon >= 1:
            return None  # nunca la alcanza
        return -np.log(razon) / self.k

    # ---------- Gráficas ----------
    def graficar_series(self, ax=None, mostrar=True):
        if ax is None:
            plt.figure(figsize=(10, 4))
            ax = plt.gca()
        ax.plot(self.tiempo, self.temperatura, color="tab:red", label=f"T(t), T0 = {self.T0}")
        ax.axhline(self.T_amb, color="gray", linestyle="--", label=f"T ambiente = {self.T_amb}")
        ax.set_xlabel("Tiempo (min)")
        ax.set_ylabel("Temperatura (°C)")
        ax.set_title(f"Evolución temporal de la temperatura (k = {self.k})")
        ax.legend()
        ax.grid(True)
        if mostrar:
            plt.show()

    def campo_de_direcciones(self, ax, t_max=30, T_min=0, T_max=100, densidad=20):
        """Campo de direcciones en el plano (t, T): cada flecha tiene pendiente dT/dt."""
        t_g, T_g = np.meshgrid(np.linspace(0, t_max, densidad),
                               np.linspace(T_min, T_max, densidad))
        pend = self.edo(t_g, T_g)
        # Se normalizan las flechas respetando la escala de los ejes
        escala_x, escala_y = t_max, (T_max - T_min)
        u = np.ones_like(pend) / escala_x
        v = pend / escala_y
        norma = np.hypot(u, v)
        ax.quiver(t_g, T_g, u / norma, v / norma, angles="xy", color="gray", alpha=0.6)
        ax.set_xlim(0, t_max)
        ax.set_ylim(T_min, T_max)

    def linea_de_fase(self, ax, T_min=0, T_max=100):
        """Línea de fase: dT/dt en función de T. Los cortes con 0 son equilibrios."""
        T = np.linspace(T_min, T_max, 200)
        ax.plot(T, self.edo(0, T), color="tab:blue")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.plot(self.T_equilibrio(), 0, "o", color="tab:red", label=f"Equilibrio T = {self.T_equilibrio():.1f}")
        ax.set_xlabel("T (°C)")
        ax.set_ylabel("dT/dt")
        ax.set_title("Línea de fase")
        ax.legend()
        ax.grid(True)

    def graficar_semilog(self, T0, tiempo_final=30, ax=None, mostrar=True):
        """ln|T - T*| contra t: una recta de pendiente -k. Sirve para estimar k con datos reales."""
        if ax is None:
            plt.figure(figsize=(8, 4))
            ax = plt.gca()
        t = np.linspace(0, tiempo_final, 400)
        dif = np.abs(self.solucion_exacta(T0, t) - self.T_equilibrio())
        if np.allclose(dif, 0):
            ax.text(0.5, 0.5, "T0 = equilibrio\n(no hay diferencia que graficar)", ha="center", va="center",
                    transform=ax.transAxes)
        else:
            ax.semilogy(t, dif, label=f"|T - T*|  (pendiente de ln = -k = {-self.k})")
            ax.legend()
        ax.set_xlabel("Tiempo (min)")
        ax.set_ylabel("|T - T*| (°C, escala log)")
        ax.set_title("Escala semilog: una recta")
        ax.grid(True, which="both", alpha=0.5)
        if mostrar:
            plt.show()

    def graficar_todo(self, T0=90.0, tiempo_final=30, h=0.5, T_min=None, T_max=None, guardar=None):
        """Panel de 6 gráficas del escenario: series, campo de direcciones, línea de fase,
        velocidad, escala semilog y error de Euler / RK4."""
        self.simular(T0, tiempo_final, paso=0.01)
        T_eq = self.T_equilibrio()
        tau = self.constante_de_tiempo()

        lo, hi = min(T0, T_eq), max(T0, T_eq)
        margen = 0.15 * max(hi - lo, 1.0)
        T_min = lo - margen if T_min is None else T_min
        T_max = hi + margen if T_max is None else T_max
        sentido = "se enfría" if T0 > T_eq else ("se calienta" if T0 < T_eq else "ya está en equilibrio")

        fig, axs = plt.subplots(2, 3, figsize=(18, 9.5))
        fig.suptitle(f"Escenario: k = {self.k}, T* = {T_eq:.1f} °C, T0 = {T0} °C  ({sentido})  |  "
                     f"τ = 1/k = {tau:.2f} min", fontsize=13)

        # 1) Series de tiempo con los tres métodos
        ax = axs[0, 0]
        ax.plot(self.tiempo, self.temperatura, color="tab:red", linewidth=2, label="RK45")
        t_e, T_e = self.euler(T0, tiempo_final, h)
        t_r, T_r = self.rk4(T0, tiempo_final, h)
        ax.plot(t_e, T_e, "o--", markersize=3, label=f"Euler (h = {h})")
        ax.plot(t_r, T_r, "s:", markersize=3, label=f"RK4 (h = {h})")
        ax.axhline(T_eq, color="gray", linestyle="--", label=f"Equilibrio = {T_eq:.1f}")
        if tau <= tiempo_final:
            T_tau = T_eq + (T0 - T_eq) / np.e
            ax.plot(tau, T_tau, "ko")
            ax.annotate("t = τ\n(63.2 %)", (tau, T_tau), textcoords="offset points", xytext=(10, 10))
        ax.set_xlabel("Tiempo (min)")
        ax.set_ylabel("Temperatura (°C)")
        ax.set_title("Series de tiempo")
        ax.legend()
        ax.grid(True)

        # 2) Campo de direcciones + trayectoria
        ax = axs[0, 1]
        self.campo_de_direcciones(ax, t_max=tiempo_final, T_min=T_min, T_max=T_max)
        ax.plot(self.tiempo, self.temperatura, color="tab:red", linewidth=2)
        ax.set_xlabel("Tiempo (min)")
        ax.set_ylabel("Temperatura (°C)")
        ax.set_title("Campo de direcciones")
        ax.grid(True)

        # 3) Línea de fase
        ax = axs[0, 2]
        self.linea_de_fase(ax, T_min=T_min, T_max=T_max)
        ax.plot(self.temperatura, self.edo(self.tiempo, self.temperatura), color="tab:red", linewidth=3, alpha=0.6,
                label="Trayectoria")
        ax.legend()

        # 4) Velocidad de cambio
        ax = axs[1, 0]
        ax.plot(self.tiempo, self.edo(self.tiempo, self.temperatura), color="tab:green")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xlabel("Tiempo (min)")
        ax.set_ylabel("dT/dt (°C/min)")
        ax.set_title("Velocidad de cambio")
        ax.grid(True)

        # 5) Escala semilog
        self.graficar_semilog(T0, tiempo_final, ax=axs[1, 1], mostrar=False)

        # 6) Error de Euler y RK4 contra la solución exacta
        ax = axs[1, 2]
        err_e = np.abs(T_e - self.solucion_exacta(T0, t_e)) + 1e-16  # t=0 tiene error 0 y se omite al graficar
        err_r = np.abs(T_r - self.solucion_exacta(T0, t_r)) + 1e-16
        ax.semilogy(t_e[1:], err_e[1:], "o-", markersize=3, label=f"Euler (máx = {err_e.max():.2e})")
        ax.semilogy(t_r[1:], err_r[1:], "s-", markersize=3, label=f"RK4 (máx = {err_r.max():.2e})")
        ax.set_xlabel("Tiempo (min)")
        ax.set_ylabel("|error| (°C)")
        ax.set_title("Error numérico")
        ax.legend()
        ax.grid(True, which="both", alpha=0.5)

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        if guardar:
            plt.savefig(guardar, dpi=150, bbox_inches="tight")
        plt.show()

## 3. Punto de equilibrio y estabilidad

Se busca $f(T)=0$:

$$-k(T - T_a) = 0 \;\Rightarrow\; T^* = T_a$$

El jacobiano de una ecuación escalar es $f'(T) = -k$, así que el único autovalor es $\lambda = -k < 0$ para todo $k>0$.
Por tanto, $T^* = T_a$ es **asintóticamente estable**: sin importar la temperatura inicial, el objeto tiende a la temperatura del ambiente.

In [ ]:
modelo = Escenario(k=k_base, T_amb=Ta_base)
modelo.simular(T0=T0_base, tiempo_final=tiempo_final, paso=paso)

equilibrios = modelo.calcular_puntos_equilibrio()

print("Puntos de equilibrio y sus autovalores:")
for eq in equilibrios:
    print(f"Punto de equilibrio: {eq['punto_equilibrio']}")
    print(f"Jacobiano: {eq['jacobiano']}")
    print(f"Autovalores: {eq['autovalores']}")
    print(f"Estabilidad: {eq['estabilidad']}\n")

**Solución analítica (sympy)** — separación de variables / factor integrante:

$$T(t) = T_a + (T_0 - T_a)\,e^{-kt}$$

In [ ]:
modelo.solucion_simbolica(T0=T0_base)

In [ ]:
t_obj = modelo.tiempo_para_alcanzar(T0=T0_base, T_objetivo=T_objetivo)
if t_obj is None:
    print(f"El objeto nunca llega a {T_objetivo} °C (está fuera del rango entre T0 y el equilibrio)")
else:
    print(f"El objeto pasa de {T0_base} °C a {T_objetivo} °C en {t_obj:.2f} min")
print(f"Vida media térmica (tiempo para cerrar la mitad de la brecha): {np.log(2)/modelo.k:.2f} min")

## 4. Casos de estudio

Se usa el mismo ambiente ($T_a = 20$ °C, $k = 0.15$) con diferentes temperaturas iniciales, y un caso con otro $k$.

| Caso | $T_0$ | $T_a$ | $k$ | Descripción |
|---|---|---|---|---|
| 1 | 90 | 20 | 0.15 | Café caliente que se enfría |
| 2 | 5 | 20 | 0.15 | Bebida fría que se calienta |
| 3 | 20 | 20 | 0.15 | Objeto ya en equilibrio |
| 4 | 90 | 20 | 0.6 | Mismo café con mejor transferencia de calor (k alto) |

In [ ]:
caso1 = Escenario(k=k_base, T_amb=Ta_base)
caso1.simular(T0=T0_base, tiempo_final=tiempo_final, paso=paso)

caso2 = Escenario(k=k_base, T_amb=Ta_base)
caso2.simular(T0=T0_frio, tiempo_final=tiempo_final, paso=paso)

caso3 = Escenario(k=k_base, T_amb=Ta_base)
caso3.simular(T0=Ta_base, tiempo_final=tiempo_final, paso=paso)

caso4 = Escenario(k=k_alto, T_amb=Ta_base)
caso4.simular(T0=T0_base, tiempo_final=tiempo_final, paso=paso)

In [ ]:
caso1.graficar_series()

In [ ]:
caso2.graficar_series()

In [ ]:
caso3.graficar_series()

In [ ]:
caso4.graficar_series()

### Comparación de todos los casos

In [ ]:
plt.figure(figsize=(10, 5))
for i, caso in enumerate([caso1, caso2, caso3, caso4], start=1):
    plt.plot(caso.tiempo, caso.temperatura, label=f"Caso {i} (T0={caso.T0}, k={caso.k})")
plt.axhline(Ta_base, color="gray", linestyle="--", label="T ambiente")
plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Todas las trayectorias convergen a T ambiente")
plt.legend()
plt.grid(True)
plt.savefig("resources/comparacion_casos.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Campo de direcciones y línea de fase

El campo de direcciones muestra la pendiente $dT/dt$ en cada punto $(t, T)$. Las trayectorias son curvas tangentes a las flechas.
Para una EDO escalar el "retrato de fase" es la **línea de fase**: la gráfica de $dT/dt$ contra $T$.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Campo de direcciones + trayectorias
caso1.campo_de_direcciones(axs[0], t_max=tiempo_final, T_min=T_min_graf, T_max=T_max_graf)
for nombre, caso in [(f"T0 = {c.T0}", c) for c in (caso1, caso2, caso3)]:
    axs[0].plot(caso.tiempo, caso.temperatura, linewidth=2, label=nombre)
axs[0].set_xlabel("Tiempo (min)")
axs[0].set_ylabel("Temperatura (°C)")
axs[0].set_title("Campo de direcciones y trayectorias")
axs[0].legend()
axs[0].grid(True)

# Línea de fase
caso1.linea_de_fase(axs[1], T_min=T_min_graf, T_max=T_max_graf)

plt.tight_layout()
plt.savefig("resources/campo_direcciones.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Método de Euler vs. solución exacta

Se compara Euler con distintos tamaños de paso $h$ frente a la solución exacta.

In [ ]:
T0 = T0_base
modelo = Escenario(k=k_base, T_amb=Ta_base)
t_fino = np.linspace(0, tiempo_final, 1000)

plt.figure(figsize=(10, 5))
plt.plot(t_fino, modelo.solucion_exacta(T0, t_fino), "k", linewidth=2, label="Solución exacta")
for h in h_lista:
    t_e, T_e = modelo.euler(T0, tiempo_final=tiempo_final, h=h)
    plt.plot(t_e, T_e, "o-", markersize=4, label=f"Euler h = {h}")
plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Método de Euler vs. solución exacta")
plt.legend()
plt.grid(True)
plt.savefig("resources/euler_vs_exacta.png", dpi=150, bbox_inches="tight")
plt.show()

### Error de Euler en función de $h$

In [ ]:
print(f"{'h':>6} | {'Error máx (°C)':>15}")
print("-" * 25)
errores = []
hs = [4.0, 2.0, 1.0, 0.5, 0.25, 0.125, 0.0625]
for h in hs:
    t_e, T_e = modelo.euler(T0, tiempo_final=32, h=h)
    err = np.max(np.abs(T_e - modelo.solucion_exacta(T0, t_e)))
    errores.append(err)
    print(f"{h:>6} | {err:>15.5f}")

plt.figure(figsize=(7, 4))
plt.loglog(hs, errores, "o-")
plt.xlabel("Paso h")
plt.ylabel("Error máximo (°C)")
plt.title("El error de Euler decrece linealmente con h (orden 1)")
plt.grid(True, which="both")
plt.show()

### Estabilidad numérica de Euler

Para esta EDO, Euler da $T_{n+1} - T_a = (1 - hk)(T_n - T_a)$. Por eso:

- Si $hk < 1$: la solución numérica decrece monótonamente (comportamiento correcto).
- Si $1 < hk < 2$: oscila pero converge (comportamiento incorrecto, aunque estable).
- Si $hk > 2$: **diverge** aunque la solución real converge.

Con $k = 0.15$ el límite de estabilidad es $h = 2/k \approx 13.3$. Para verlo con pasos más pequeños, se usa $k = 0.6$ (límite $h \approx 3.33$).

In [ ]:
modelo_rapido = Escenario(k=k_alto, T_amb=Ta_base)
t_fino = np.linspace(0, 20, 1000)

plt.figure(figsize=(10, 5))
plt.plot(t_fino, modelo_rapido.solucion_exacta(T0_base, t_fino), "k", linewidth=2, label="Exacta")
for h, estilo in [(0.6 / k_alto, "hk = 0.6 (monótona)"), (1.5 / k_alto, "hk = 1.5 (oscila y converge)"), (2.16 / k_alto, "hk = 2.16 (diverge)")]:
    t_e, T_e = modelo_rapido.euler(T0_base, tiempo_final=20, h=h)
    plt.plot(t_e, T_e, "o-", markersize=4, label=f"h = {h:.2f}: {estilo}")
plt.ylim(-100, 200)
plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Estabilidad del método de Euler (k = 0.6)")
plt.legend()
plt.grid(True)
plt.savefig("resources/estabilidad_euler.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Modelo modificado 1: temperatura ambiente variable

En la vida real el ambiente no es constante. Se modela el ciclo día–noche con una senoidal:

$$\frac{dT}{dt} = -k\,\bigl(T - T_a(t)\bigr), \qquad T_a(t) = T_m + A\sin\!\left(\frac{2\pi t}{P}\right)$$

- $T_m$: temperatura media del ambiente
- $A$: amplitud de la oscilación
- $P$: periodo (24 h)

Ahora **no existe un punto de equilibrio fijo**: el objeto sigue al ambiente con un retraso (desfase) y menor amplitud.

In [ ]:
class EscenarioVariable(Escenario):
    """dT/dt = -k (T - T_amb(t)), con T_amb(t) = T_m + A sin(2 pi t / P)."""

    def __init__(self, k=0.5, T_m=20.0, A=8.0, P=24.0):
        super().__init__(k=k, T_amb=T_m)
        self.T_m, self.A, self.P = T_m, A, P

    def T_ambiente(self, t):
        return self.T_m + self.A * np.sin(2 * np.pi * t / self.P)

    def edo(self, tiempo, T):
        return -self.k * (T - self.T_ambiente(tiempo))

    def graficar_series(self, ax=None, mostrar=True):
        plt.figure(figsize=(10, 4))
        plt.plot(self.tiempo, self.temperatura, color="tab:red", label=f"Objeto (T0 = {self.T0})")
        plt.plot(self.tiempo, self.T_ambiente(self.tiempo), "--", color="gray", label="Ambiente T_a(t)")
        plt.xlabel("Tiempo (h)")
        plt.ylabel("Temperatura (°C)")
        plt.title("Objeto en un ambiente con ciclo día–noche")
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
variable = EscenarioVariable(k=k_var, T_m=Tm_var, A=A_var, P=P_var)
variable.simular(T0=T0_var, tiempo_final=tiempo_var, paso=paso)
variable.graficar_series()

**Amplitud y desfase en régimen permanente.** Para $T_a(t)=T_m + A\sin(\omega t)$, la solución estacionaria es
$T_{ss}(t)=T_m + \dfrac{A}{\sqrt{1+(\omega/k)^2}}\sin(\omega t - \varphi)$, con $\varphi = \arctan(\omega/k)$.

In [ ]:
omega = 2 * np.pi / variable.P
amp_teorica = variable.A / np.sqrt(1 + (omega / variable.k) ** 2)
desfase_horas = np.arctan(omega / variable.k) / omega

# Amplitud numérica: último periodo de la simulación
ult = variable.tiempo > variable.tiempo_final - variable.P
amp_numerica = (variable.temperatura[ult].max() - variable.temperatura[ult].min()) / 2

print(f"Amplitud teórica del objeto:  {amp_teorica:.3f} °C")
print(f"Amplitud numérica del objeto: {amp_numerica:.3f} °C")
print(f"Retraso respecto al ambiente: {desfase_horas:.2f} h")

## 8. Modelo modificado 2: sistema acoplado objeto–ambiente

Si el ambiente (por ejemplo, una habitación pequeña) también se calienta con el objeto, ya no se puede suponer $T_a$ constante.
Se tiene un **sistema de dos ecuaciones**, parecido al de presa–depredador:

$$\frac{dT}{dt} = -k_1\,(T - T_a), \qquad \frac{dT_a}{dt} = k_2\,(T - T_a) - k_3\,(T_a - T_{ext})$$

- $T$: temperatura del objeto, $T_a$: temperatura del aire de la habitación
- $k_1$: transferencia objeto → aire, $k_2$: calentamiento del aire por el objeto
- $k_3$: pérdida de calor del aire hacia el exterior (a temperatura $T_{ext}$ constante)

In [ ]:
class EscenarioAcoplado:
    def __init__(self, k1=0.3, k2=0.1, k3=0.05, T_ext=15.0):
        self.k1, self.k2, self.k3, self.T_ext = k1, k2, k3, T_ext
        self.T_symb, self.Ta_symb = sp.symbols("T T_a", real=True)
        self.f1 = -self.k1 * (self.T_symb - self.Ta_symb)
        self.f2 = self.k2 * (self.T_symb - self.Ta_symb) - self.k3 * (self.Ta_symb - self.T_ext)

    def sistema_edo(self, tiempo, estado):
        T, Ta = estado
        return [-self.k1 * (T - Ta),
                self.k2 * (T - Ta) - self.k3 * (Ta - self.T_ext)]

    def simular(self, T0, Ta0, tiempo_final=80, paso=0.01):
        self.T0, self.Ta0 = T0, Ta0
        t_eval = np.linspace(0, tiempo_final, int(tiempo_final / paso) + 1)
        sol = solve_ivp(self.sistema_edo, [0, tiempo_final], [T0, Ta0], t_eval=t_eval, method="RK45")
        self.tiempo, self.temperatura, self.temp_aire = sol.t, sol.y[0], sol.y[1]
        return self.tiempo, self.temperatura, self.temp_aire

    def calcular_puntos_equilibrio(self):
        puntos = sp.solve([self.f1, self.f2], (self.T_symb, self.Ta_symb), dict=True)
        J = sp.Matrix([self.f1, self.f2]).jacobian([self.T_symb, self.Ta_symb])
        resultados = []
        for p in puntos:
            T_eq, Ta_eq = float(p[self.T_symb]), float(p[self.Ta_symb])
            autoval = {complex(round(float(sp.re(ev)), 3), round(float(sp.im(ev)), 3)): m
                       for ev, m in J.eigenvals().items()}
            resultados.append({"punto_equilibrio": (round(T_eq, 2), round(Ta_eq, 2)),
                               "jacobiano": [[round(float(J[i, j]), 3) for j in range(2)] for i in range(2)],
                               "autovalores": autoval})
        return resultados

    def graficar_series(self):
        plt.figure(figsize=(10, 4))
        plt.plot(self.tiempo, self.temperatura, color="tab:red", label=f"Objeto (T0 = {self.T0})")
        plt.plot(self.tiempo, self.temp_aire, color="tab:blue", label=f"Aire (Ta0 = {self.Ta0})")
        plt.axhline(self.T_ext, color="gray", linestyle="--", label=f"Exterior = {self.T_ext}")
        plt.xlabel("Tiempo (min)")
        plt.ylabel("Temperatura (°C)")
        plt.title("Sistema acoplado objeto–aire–exterior")
        plt.legend()
        plt.grid(True)
        plt.show()

    def retrato_de_fase(self, ax=None, paso=400):
        if ax is None:
            ax = plt.gca()
        ax.plot(self.temperatura, self.temp_aire, label=f"T0 = {self.T0}, Ta0 = {self.Ta0}")
        # Flechas con la dirección del campo vectorial en puntos de la trayectoria
        xs = self.temperatura[0:-1:paso]
        ys = self.temp_aire[0:-1:paso]
        dxs = -self.k1 * (xs - ys)
        dys = self.k2 * (xs - ys) - self.k3 * (ys - self.T_ext)
        norma = np.hypot(dxs, dys) + 1e-12
        ax.quiver(xs, ys, dxs / norma, dys / norma, angles="xy", width=0.004, zorder=0)

In [ ]:
acoplado = EscenarioAcoplado(k1=k1_ac, k2=k2_ac, k3=k3_ac, T_ext=T_ext_ac)
acoplado.simular(T0=T0_ac, Ta0=Ta0_ac, tiempo_final=tiempo_ac)

print("Puntos de equilibrio y sus autovalores:")
for eq in acoplado.calcular_puntos_equilibrio():
    print(f"Punto de equilibrio: {eq['punto_equilibrio']}")
    print(f"Matriz Jacobiana: {eq['jacobiano']}")
    print(f"Autovalores: {eq['autovalores']}\n")

acoplado.graficar_series()

Los dos autovalores son reales y negativos, así que el equilibrio $(T_{ext}, T_{ext})$ es un **nodo estable**: objeto y aire terminan igualando la temperatura exterior.
Nótese que, a diferencia del modelo presa–depredador, **no hay oscilaciones** (no hay autovalores complejos).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for T0_i, Ta0_i in cond_iniciales_ac:
    caso = EscenarioAcoplado(k1=k1_ac, k2=k2_ac, k3=k3_ac, T_ext=T_ext_ac)
    caso.simular(T0=T0_i, Ta0=Ta0_i, tiempo_final=tiempo_ac)
    caso.retrato_de_fase(ax)
ax.plot(T_ext_ac, T_ext_ac, "ko", label=f"Equilibrio ({T_ext_ac}, {T_ext_ac})")
ax.set_xlabel("Temperatura del objeto T")
ax.set_ylabel("Temperatura del aire T_a")
ax.set_title("Retrato de fase del sistema acoplado")
ax.legend()
ax.grid(True)
plt.savefig("resources/retrato_fase_acoplado.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Variaciones de los parámetros

Se analiza cómo cambia el comportamiento al variar cada parámetro por separado. Las listas se editan en la celda de **Parámetros globales**. Se usa la solución exacta, que coincide con la simulación numérica (RK45).

In [ ]:
t_plot = np.linspace(0, tiempo_final, 600)
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True)

# (a) Variación de k
for k_i in k_lista:
    e = Escenario(k=k_i, T_amb=Ta_base)
    axs[0].plot(t_plot, e.solucion_exacta(T0_base, t_plot), label=f"k = {k_i}")
axs[0].set_title(f"(a) Variación de k  (T0 = {T0_base}, Ta = {Ta_base})")

# (b) Variación de la temperatura ambiente
for Ta_i in Ta_lista:
    e = Escenario(k=k_base, T_amb=Ta_i)
    linea, = axs[1].plot(t_plot, e.solucion_exacta(T0_base, t_plot), label=f"Ta = {Ta_i}")
    axs[1].axhline(Ta_i, color=linea.get_color(), linestyle=":", alpha=0.6)
axs[1].set_title(f"(b) Variación de Ta  (T0 = {T0_base}, k = {k_base})")

# (c) Variación de la temperatura inicial
for T0_i in T0_lista:
    e = Escenario(k=k_base, T_amb=Ta_base)
    axs[2].plot(t_plot, e.solucion_exacta(T0_i, t_plot), label=f"T0 = {T0_i}")
axs[2].axhline(Ta_base, color="gray", linestyle="--")
axs[2].set_title(f"(c) Variación de T0  (Ta = {Ta_base}, k = {k_base})")

for ax in axs:
    ax.set_xlabel("Tiempo (min)")
    ax.legend()
    ax.grid(True)
axs[0].set_ylabel("Temperatura (°C)")
plt.tight_layout()
plt.savefig("resources/variaciones_parametros.png", dpi=150, bbox_inches="tight")
plt.show()

### Velocidad de cambio $dT/dt$

El objeto cambia de temperatura más rápido al principio (cuando la diferencia con el ambiente es mayor) y se frena al acercarse al equilibrio.

In [ ]:
plt.figure(figsize=(10, 4.5))
for k_i in k_lista[::2]:
    e = Escenario(k=k_i, T_amb=Ta_base)
    T = e.solucion_exacta(T0_base, t_plot)
    plt.plot(t_plot, e.edo(t_plot, T), label=f"k = {k_i}")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Tiempo (min)")
plt.ylabel("dT/dt (°C/min)")
plt.title("Velocidad de cambio de la temperatura en el tiempo")
plt.legend()
plt.grid(True)
plt.show()

### Tiempo necesario para llegar a una temperatura objetivo

Despejando de la solución exacta: $t = -\dfrac{1}{k}\ln\!\left(\dfrac{T_{obj}-T_a}{T_0-T_a}\right)$.
Solo tiene sentido si la temperatura objetivo está **entre** $T_0$ y $T_a$ (el equilibrio nunca se alcanza exactamente).

In [ ]:
ks = np.linspace(0.02, 1.0, 200)
plt.figure(figsize=(10, 4.5))
for T_obj in T_obj_lista:
    razon = (T_obj - Ta_base) / (T0_base - Ta_base)
    if 0 < razon < 1:
        plt.plot(ks, -np.log(razon) / ks, label=f"Llegar a {T_obj} °C")
    else:
        print(f"Se omite T_obj = {T_obj}: no está entre T0 = {T0_base} y Ta = {Ta_base}")
plt.ylim(0, 60)
plt.xlabel("k (1/min)")
plt.ylabel("Tiempo (min)")
plt.title(f"Tiempo de cambio vs. k  (T0 = {T0_base}, Ta = {Ta_base})")
plt.legend()
plt.grid(True)
plt.savefig("resources/tiempo_vs_k.png", dpi=150, bbox_inches="tight")
plt.show()

### Campos de direcciones para distintos valores de $k$

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)
for ax, k_i in zip(axs.ravel(), k_campos):
    e = Escenario(k=k_i, T_amb=Ta_base)
    e.campo_de_direcciones(ax, t_max=tiempo_final, T_min=T_min_graf, T_max=T_max_graf)
    for T0_i in [T0_base, (T0_base + Ta_base) / 2, T0_frio]:
        ax.plot(t_plot, e.solucion_exacta(T0_i, t_plot), linewidth=2, label=f"T0 = {T0_i:g}")
    ax.set_title(f"k = {k_i}")
    ax.grid(True)
    ax.legend(loc="upper right")
for ax in axs[1]:
    ax.set_xlabel("Tiempo (min)")
for ax in axs[:, 0]:
    ax.set_ylabel("Temperatura (°C)")
plt.tight_layout()
plt.savefig("resources/campos_direcciones_k.png", dpi=150, bbox_inches="tight")
plt.show()

### Mapa de calor: temperatura en un instante

Muestra $T(t_{mapa})$ como función de $k$ y de la temperatura inicial $T_0$ (con $T_a$ de los parámetros globales). Las curvas de nivel indican qué combinaciones dan la misma temperatura.

In [ ]:
k_g, T0_g = np.meshgrid(np.linspace(0.01, 0.6, 200), np.linspace(T_min_graf, T_max_graf, 200))
T_mapa = Ta_base + (T0_g - Ta_base) * np.exp(-k_g * t_mapa)

plt.figure(figsize=(9, 6))
mapa = plt.contourf(k_g, T0_g, T_mapa, levels=30, cmap="coolwarm")
plt.colorbar(mapa, label=f"T({t_mapa}) (°C)")
curvas = plt.contour(k_g, T0_g, T_mapa, levels=8, colors="black", linewidths=0.8)
plt.clabel(curvas, fmt="%d °C")
plt.xlabel("k (1/min)")
plt.ylabel("T0 (°C)")
plt.title(f"Temperatura a los {t_mapa} min según k y T0  (Ta = {Ta_base})")
plt.savefig("resources/mapa_calor.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Modelo modificado 3: fuente de calor constante

Si el objeto tiene una fuente de calor (una resistencia, un calentador, una reacción), se agrega un término constante $q$ (°C/min):

$$\frac{dT}{dt} = -k\,(T - T_a) + q$$

El equilibrio se desplaza a $T^* = T_a + \dfrac{q}{k}$ y sigue siendo estable ($\lambda = -k$). Ahora **el equilibrio ya no coincide con el ambiente**.

In [ ]:
class EscenarioFuente(Escenario):
    # dT/dt = -k (T - T_amb) + q

    def __init__(self, k=0.15, T_amb=20.0, q=3.0):
        super().__init__(k=k, T_amb=T_amb)
        self.q = q
        self.ecuacion = -self.k * (self.T_symb - self.T_amb) + self.q

    def edo(self, tiempo, T):
        return -self.k * (T - self.T_amb) + self.q

    def T_equilibrio(self):
        return self.T_amb + self.q / self.k

    def solucion_exacta(self, T0, t):
        T_eq = self.T_equilibrio()
        return T_eq + (T0 - T_eq) * np.exp(-self.k * t)

In [ ]:
fuente = EscenarioFuente(k=k_base, T_amb=Ta_base, q=q_base)
fuente.simular(T0=Ta_base, tiempo_final=40)

for eq in fuente.calcular_puntos_equilibrio():
    print(f"Equilibrio: {eq['punto_equilibrio']}  | autovalores: {eq['autovalores']}  | {eq['estabilidad']}")

fig, axs = plt.subplots(1, 2, figsize=(15, 4.8))

# Series de tiempo para distintas fuentes q
t_p = np.linspace(0, 40, 600)
for q_i in q_lista:
    e = EscenarioFuente(k=k_base, T_amb=Ta_base, q=q_i)
    linea, = axs[0].plot(t_p, e.solucion_exacta(Ta_base, t_p), label=f"q = {q_i}")
    axs[0].axhline(e.T_equilibrio(), color=linea.get_color(), linestyle=":", alpha=0.6)
axs[0].set_xlabel("Tiempo (min)")
axs[0].set_ylabel("Temperatura (°C)")
axs[0].set_title("Efecto de la fuente de calor q  (líneas punteadas = equilibrio)")
axs[0].legend()
axs[0].grid(True)

# Equilibrio en función de q y k
q_v = np.linspace(0, max(q_lista) * 1.3, 100)
for k_i in k_lista[:4]:
    axs[1].plot(q_v, Ta_base + q_v / k_i, label=f"k = {k_i}")
axs[1].set_xlabel("q (°C/min)")
axs[1].set_ylabel("Temperatura de equilibrio T* (°C)")
axs[1].set_title("T* = Ta + q/k")
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.savefig("resources/fuente_calor.png", dpi=150, bbox_inches="tight")
plt.show()

Con una fuente de calor, un valor de $k$ más pequeño (peor disipación) produce un equilibrio **más caliente**: es el principio de por qué un aparato mal ventilado se sobrecalienta.

## 11. Gráficas adicionales

### Escala semilog: estimar $k$ a partir de los datos

Como $T - T^* = (T_0 - T^*)e^{-kt}$, al graficar $\ln|T - T^*|$ contra $t$ sale una **recta de pendiente $-k$**. Es la forma estándar de obtener $k$ con mediciones de laboratorio.

### Constante de tiempo $\tau = 1/k$

Es el tiempo en que el objeto recorre el 63.2 % de la distancia hasta el equilibrio. Tras $3\tau$ ya recorrió el 95 %, y tras $5\tau$ el 99.3 %.

In [ ]:
modelo = Escenario(k=k_base, T_amb=Ta_base)
tau = modelo.constante_de_tiempo()
print(f"τ = 1/k = {tau:.2f} min")
for n_tau in [1, 2, 3, 5]:
    print(f"  t = {n_tau}τ = {n_tau * tau:5.1f} min -> recorre el {100 * (1 - np.exp(-n_tau)):.1f} % de la distancia al equilibrio")

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))

# Series con marcas de tau
t_c = np.linspace(0, tiempo_final, 500)
axs[0].plot(t_c, modelo.solucion_exacta(T0_base, t_c), color="tab:red")
axs[0].axhline(Ta_base, color="gray", linestyle="--")
for n_tau in [1, 2, 3]:
    if n_tau * tau <= tiempo_final:
        T_n = modelo.solucion_exacta(T0_base, n_tau * tau)
        axs[0].plot(n_tau * tau, T_n, "ko")
        axs[0].annotate(f"{n_tau}τ", (n_tau * tau, T_n), textcoords="offset points", xytext=(8, 8))
axs[0].set_xlabel("Tiempo (min)")
axs[0].set_ylabel("Temperatura (°C)")
axs[0].set_title("Constante de tiempo τ")
axs[0].grid(True)

# Semilog para varios k
for k_i in k_lista[:4]:
    Escenario(k=k_i, T_amb=Ta_base).graficar_semilog(T0_base, tiempo_final, ax=axs[1], mostrar=False)
axs[1].lines[0].set_label(f"k = {k_lista[0]}")
for linea, k_i in zip(axs[1].lines, k_lista):
    linea.set_label(f"k = {k_i}")
axs[1].legend()
axs[1].set_title("Semilog: pendiente = -k")

plt.tight_layout()
plt.savefig("resources/semilog_tau.png", dpi=150, bbox_inches="tight")
plt.show()

### Euler vs. Runge-Kutta de orden 4

Euler tiene error de orden 1 (el error baja a la mitad si $h$ se reduce a la mitad); RK4 tiene orden 4 (baja unas 16 veces).

In [ ]:
modelo = Escenario(k=k_base, T_amb=Ta_base)
hs = [4.0, 2.0, 1.0, 0.5, 0.25, 0.125]
err_euler, err_rk4 = [], []

print(f"{'h':>6} | {'Euler':>12} | {'RK4':>12}")
print("-" * 36)
for h in hs:
    t_e, T_e = modelo.euler(T0_base, tiempo_final=32, h=h)
    t_r, T_r = modelo.rk4(T0_base, tiempo_final=32, h=h)
    err_euler.append(np.max(np.abs(T_e - modelo.solucion_exacta(T0_base, t_e))))
    err_rk4.append(np.max(np.abs(T_r - modelo.solucion_exacta(T0_base, t_r))))
    print(f"{h:>6} | {err_euler[-1]:>12.2e} | {err_rk4[-1]:>12.2e}")

plt.figure(figsize=(8, 5))
plt.loglog(hs, err_euler, "o-", label="Euler (orden 1)")
plt.loglog(hs, err_rk4, "s-", label="RK4 (orden 4)")
plt.loglog(hs, [err_euler[0] * (h / hs[0]) for h in hs], "k:", alpha=0.5, label="pendiente 1")
plt.loglog(hs, [err_rk4[0] * (h / hs[0]) ** 4 for h in hs], "k--", alpha=0.5, label="pendiente 4")
plt.xlabel("Paso h")
plt.ylabel("Error máximo (°C)")
plt.title("Orden de convergencia de Euler y RK4")
plt.legend()
plt.grid(True, which="both")
plt.savefig("resources/euler_vs_rk4.png", dpi=150, bbox_inches="tight")
plt.show()

### Panel completo con un solo comando

`graficar_todo()` muestra en una sola figura las 6 gráficas principales de un escenario: series de tiempo (RK45, Euler y RK4), campo de direcciones, línea de fase, velocidad, escala semilog y error numérico.

In [ ]:
# Enfriamiento
Escenario(k=k_base, T_amb=Ta_base).graficar_todo(
    T0=T0_base, tiempo_final=tiempo_final, h=h_euler, guardar="resources/panel_enfriamiento.png")

In [ ]:
# Calentamiento
Escenario(k=k_base, T_amb=Ta_base).graficar_todo(
    T0=T0_frio, tiempo_final=tiempo_final, h=h_euler, guardar="resources/panel_calentamiento.png")

In [ ]:
# También funciona con el modelo de fuente de calor (el equilibrio cambia a Ta + q/k)
EscenarioFuente(k=k_base, T_amb=Ta_base, q=q_base).graficar_todo(
    T0=Ta_base, tiempo_final=40, h=h_euler, guardar="resources/panel_fuente.png")

## 12. Panel interactivo con deslizadores

Mueve los deslizadores para cambiar los parámetros y el panel se redibuja. Requiere `ipywidgets` (viene en `requirements.txt`) y funciona en Jupyter, VS Code y Google Colab.
Si no se muestran los deslizadores, ejecuta `pip install ipywidgets` y reinicia el kernel.

In [ ]:
try:
    import ipywidgets as widgets
    HAY_WIDGETS = True
except ImportError:
    HAY_WIDGETS = False
    print("ipywidgets no está instalado: ejecuta  pip install ipywidgets  y reinicia el kernel.")

In [ ]:
if HAY_WIDGETS:
    def panel_basico(k, T_amb, T0, tiempo_final, h):
        Escenario(k=k, T_amb=T_amb).graficar_todo(T0=T0, tiempo_final=tiempo_final, h=h)

    estilo = {"description_width": "initial"}
    widgets.interact(
        panel_basico,
        k=widgets.FloatSlider(value=k_base, min=0.01, max=1.5, step=0.01, description="k (1/min)", style=estilo, continuous_update=False),
        T_amb=widgets.FloatSlider(value=Ta_base, min=-10, max=60, step=1, description="T ambiente (°C)", style=estilo, continuous_update=False),
        T0=widgets.FloatSlider(value=T0_base, min=-20, max=120, step=1, description="T0 (°C)", style=estilo, continuous_update=False),
        tiempo_final=widgets.IntSlider(value=tiempo_final, min=5, max=100, step=5, description="Tiempo final (min)", style=estilo, continuous_update=False),
        h=widgets.FloatSlider(value=h_euler, min=0.05, max=5, step=0.05, description="Paso h", style=estilo, continuous_update=False),
    );

### Panel interactivo con fuente de calor

In [ ]:
if HAY_WIDGETS:
    def panel_fuente(k, T_amb, q, T0, tiempo_final, h):
        EscenarioFuente(k=k, T_amb=T_amb, q=q).graficar_todo(T0=T0, tiempo_final=tiempo_final, h=h)

    widgets.interact(
        panel_fuente,
        k=widgets.FloatSlider(value=k_base, min=0.01, max=1.5, step=0.01, description="k (1/min)", style=estilo, continuous_update=False),
        T_amb=widgets.FloatSlider(value=Ta_base, min=-10, max=60, step=1, description="T ambiente (°C)", style=estilo, continuous_update=False),
        q=widgets.FloatSlider(value=q_base, min=0, max=10, step=0.5, description="q (°C/min)", style=estilo, continuous_update=False),
        T0=widgets.FloatSlider(value=Ta_base, min=-20, max=120, step=1, description="T0 (°C)", style=estilo, continuous_update=False),
        tiempo_final=widgets.IntSlider(value=40, min=5, max=100, step=5, description="Tiempo final (min)", style=estilo, continuous_update=False),
        h=widgets.FloatSlider(value=h_euler, min=0.05, max=5, step=0.05, description="Paso h", style=estilo, continuous_update=False),
    );

## Conclusiones

- La EDO $dT/dt = -k(T-T_a)$ tiene un único equilibrio $T^*=T_a$, asintóticamente estable ($\lambda=-k$).
- La solución exacta $T_a + (T_0-T_a)e^{-kt}$ coincide con RK45; el método de Euler converge con orden 1 y es estable solo si $hk<2$.
- Con ambiente variable, el objeto sigue al ambiente con menor amplitud y un retraso temporal.
- Con ambiente acoplado, el sistema 2×2 converge a un nodo estable sin oscilaciones.
- Un valor mayor de $k$ acelera el acercamiento al equilibrio pero no lo cambia; solo $T_a$ (y $q$, si hay fuente) lo determinan.
- Con una fuente de calor $q$, el equilibrio pasa a ser $T_a + q/k$.